# Modelo B — notebook maestro de Colab (C3+C4)
Ejecutar de arriba hacia abajo. La primera celda concentra todos los parámetros editables.

In [ ]:
# PARÁMETROS EDITABLES (única celda que debe editar el usuario)
REPO_URL = 'https://github.com/OWNER/REPOSITORY.git'
GIT_REVISION = 'feature/model-b-colab-integration'  # branch, tag o commit explícito
DRIVE_BASE = '/content/drive/MyDrive/model_b'
DRIVE_MUTANTS_HDF5 = DRIVE_BASE + '/data/proc_483p.hdf5'
DRIVE_WT_HDF5 = DRIVE_BASE + '/data/wt_companion.hdf5'
DRIVE_RUNS_ROOT = DRIVE_BASE + '/runs'
EXECUTION_MODE = 'preflight'  # preflight | smoke | pilot | resume
OUTPUT_MODE = 'drive'  # drive | local_sync
REQUESTED_DEVICE = 'auto'  # auto | cpu | cuda
SEED = 123
PILOT_EPOCHS = 3
PILOT_BATCH_SIZE = 4
RESUME_CHECKPOINT = ''  # checkpoint explícito; nunca se autoselecciona
RUN_PILOT = False
RUN_RESUME = False
SYNC_LOCAL_OUTPUTS = False
LOCAL_ROOT = '/content/model_b_workspace'
REPO_DIR = LOCAL_ROOT + '/repo'
STAGING_ROOT = LOCAL_ROOT + '/staging'
LOCAL_RUNS_ROOT = LOCAL_ROOT + '/runs'
VALID_EXECUTION_MODES = {'preflight', 'smoke', 'pilot', 'resume'}
if EXECUTION_MODE not in VALID_EXECUTION_MODES: raise ValueError(f'EXECUTION_MODE inválido: {EXECUTION_MODE!r}')
if RUN_PILOT != (EXECUTION_MODE == 'pilot'): raise ValueError('RUN_PILOT debe ser True únicamente con EXECUTION_MODE=pilot')
if RUN_RESUME != (EXECUTION_MODE == 'resume'): raise ValueError('RUN_RESUME debe ser True únicamente con EXECUTION_MODE=resume')


## 1. Montaje y validación de Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
drive_base = Path(DRIVE_BASE).resolve()
if not drive_base.is_dir(): raise FileNotFoundError(f'No existe DRIVE_BASE: {drive_base}')
for label, raw in [('mutants', DRIVE_MUTANTS_HDF5), ('WT companion', DRIVE_WT_HDF5)]:
    path = Path(raw).resolve()
    if not path.is_file(): raise FileNotFoundError(f'Falta HDF5 {label}: {path}')
    print(label, path.name, path.stat().st_size, 'bytes')


## 2. Checkout reproducible en un clon controlado

In [ ]:
import subprocess
repo = Path(REPO_DIR).resolve(); local_root = Path(LOCAL_ROOT).resolve()
if not repo.is_relative_to(local_root) or repo == local_root: raise ValueError('REPO_DIR fuera de LOCAL_ROOT')
local_root.mkdir(parents=True, exist_ok=True)
if not (repo / '.git').is_dir():
    if repo.exists(): raise RuntimeError(f'REPO_DIR existe pero no es el clon controlado: {repo}')
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(repo)], check=True)
else:
    origin = subprocess.run(['git','-C',str(repo),'remote','get-url','origin'], check=True, capture_output=True, text=True).stdout.strip()
    if origin != REPO_URL: raise RuntimeError(f'Origin inesperado: {origin}')
import re, sys
origin = subprocess.run(['git','-C',str(repo),'remote','get-url','origin'], check=True, capture_output=True, text=True).stdout.strip()
if origin != REPO_URL: raise RuntimeError(f'Origin inesperado: {origin}')
dirty = subprocess.run(['git','-C',str(repo),'status','--porcelain'], check=True, capture_output=True, text=True).stdout.strip()
if dirty: raise RuntimeError('El clon controlado está sucio; se detiene sin borrar nada')
subprocess.run(['git','-C',str(repo),'fetch','--tags','--prune','origin'], check=True)
if re.fullmatch(r'[0-9a-fA-F]{7,40}', GIT_REVISION):
    resolved_ref = GIT_REVISION
else:
    branch_name = GIT_REVISION.removeprefix('origin/'); remote_ref = 'refs/remotes/origin/' + branch_name; tag_ref = 'refs/tags/' + GIT_REVISION
    branch_exists = subprocess.run(['git','-C',str(repo),'show-ref','--verify','--quiet',remote_ref]).returncode == 0
    tag_exists = subprocess.run(['git','-C',str(repo),'show-ref','--verify','--quiet',tag_ref]).returncode == 0
    if branch_exists and tag_exists: raise RuntimeError(f'Revisión Git ambigua (rama remota y tag): {GIT_REVISION}')
    if not branch_exists and not tag_exists: raise RuntimeError(f'Revisión Git inexistente: {GIT_REVISION}')
    resolved_ref = remote_ref if branch_exists else tag_ref
GIT_COMMIT = subprocess.run(['git','-C',str(repo),'rev-parse','--verify',resolved_ref+'^{commit}'], check=True, capture_output=True, text=True).stdout.strip()
subprocess.run(['git','-C',str(repo),'checkout','--detach',GIT_COMMIT], check=True)
if subprocess.run(['git','-C',str(repo),'rev-parse','HEAD'], check=True, capture_output=True, text=True).stdout.strip() != GIT_COMMIT: raise RuntimeError('Checkout Git no coincide con el SHA resuelto')
sys.path.insert(0, str(repo / 'src')); sys.path.insert(0, str(repo))
from scripts.colab_preflight import git_revision
git_revision(repo, GIT_COMMIT)
GIT_INFO = {'requested_revision': GIT_REVISION, 'resolved_ref': resolved_ref, 'commit': GIT_COMMIT, 'remote_url': origin, 'working_tree': 'clean'}
print(GIT_INFO)


## 3. Instalación idempotente e información del runtime

In [ ]:
import json, platform, sys
req = repo / 'requirements-colab.txt'
if sys.version_info < (3, 10): raise RuntimeError('El proyecto requiere Python >=3.10')
try:
    import torch
except ImportError as exc:
    raise RuntimeError('El runtime debe proporcionar PyTorch; no se instalará una variante CUDA a ciegas') from exc
from scripts.colab_preflight import prepare_colab_environment, git_revision
ENVIRONMENT = prepare_colab_environment(repo_root=repo, marker_root=local_root, commit=GIT_COMMIT, requirements_path=req, device=REQUESTED_DEVICE, python_version=platform.python_version(), torch_version=torch.__version__)
RUNTIME = ENVIRONMENT['runtime']
git_revision(repo, GIT_COMMIT)
print({'environment_marker': ENVIRONMENT['marker'], 'reused': ENVIRONMENT['reused']})
print(json.dumps(RUNTIME, indent=2, default=str))


## 4. Staging local verificado de los HDF5

In [ ]:
from scripts.colab_preflight import stage_file, require_free_space
staging = Path(STAGING_ROOT).resolve(); staging.mkdir(parents=True, exist_ok=True)
required = Path(DRIVE_MUTANTS_HDF5).stat().st_size + Path(DRIVE_WT_HDF5).stat().st_size
require_free_space(staging, required * 2)
MUTANTS_RECORD = stage_file(DRIVE_MUTANTS_HDF5, staging/'mutants.hdf5', staging_root=staging, role='mutants')
WT_RECORD = stage_file(DRIVE_WT_HDF5, staging/'wt_companion.hdf5', staging_root=staging, role='wt_companion')
HDF5_LOCATORS = [{key: record[key] for key in ('role','drive_locator','local_locator','sha256','size_bytes','reused')} for record in (MUTANTS_RECORD, WT_RECORD)]
print(json.dumps(HDF5_LOCATORS, indent=2))


## 5. Outputs y configuración resuelta (solo overrides operacionales)

In [ ]:
from scripts.colab_preflight import generate_runtime_config, preflight_output_root, build_train_command, write_session_record, confined_path
if OUTPUT_MODE not in {'drive','local_sync'}: raise ValueError('OUTPUT_MODE debe ser drive o local_sync')
DRIVE_SYNC_ROOT = confined_path(DRIVE_RUNS_ROOT, drive_base, label='Drive runs root')
OUTPUT_ROOT = DRIVE_SYNC_ROOT if OUTPUT_MODE == 'drive' else Path(LOCAL_RUNS_ROOT).resolve()
ALLOWED_OUTPUT_ROOT = drive_base if OUTPUT_MODE == 'drive' else local_root
OUTPUT_PREFLIGHT = preflight_output_root(OUTPUT_ROOT, allowed_root=ALLOWED_OUTPUT_ROOT)
resolved_dir = local_root / 'resolved_configs'; resolved_dir.mkdir(parents=True, exist_ok=True)
base_config = repo / 'configs' / ('model_b_end_to_end_smoke.yaml' if EXECUTION_MODE == 'smoke' else 'model_b_baseline.yaml')
RESOLVED_CONFIG = resolved_dir / (EXECUTION_MODE + '.yaml')
CONFIG_RESULT = generate_runtime_config(base_config, RESOLVED_CONFIG, overrides={
 'paths.mutants_hdf5': MUTANTS_RECORD['local_locator'], 'paths.wt_companion_hdf5': WT_RECORD['local_locator'],
 'outputs.root_dir': str(OUTPUT_ROOT), 'split.persist_path': str(OUTPUT_ROOT/'splits'/'leave_position_out.json'), 'training.device': REQUESTED_DEVICE, 'training.epochs': (1 if EXECUTION_MODE == 'smoke' else PILOT_EPOCHS),
 'training.batch_size': (2 if EXECUTION_MODE == 'smoke' else PILOT_BATCH_SIZE), 'project.seed': SEED})
print(json.dumps(CONFIG_RESULT['changes'], indent=2, default=str))
from datetime import datetime, timezone
SESSION_RECORD = {'commit': GIT_COMMIT, 'requested_revision': GIT_REVISION, 'runtime': RUNTIME, 'hdf5': HDF5_LOCATORS, 'config': str(RESOLVED_CONFIG), 'seed': SEED, 'commands': []}
SESSION_RECORD_PATH = OUTPUT_ROOT/'colab_sessions'/('session-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')+'.json')
write_session_record(SESSION_RECORD_PATH, allowed_root=OUTPUT_ROOT, payload=SESSION_RECORD)


## 6. Preflight de datos, fingerprints, schema, device y resume explícito

In [ ]:
from scripts.colab_preflight import validate_runtime_hdf5
HDF5_VALIDATION = validate_runtime_hdf5(RESOLVED_CONFIG)
if EXECUTION_MODE == 'resume' and not RESUME_CHECKPOINT: raise ValueError('Resume exige RESUME_CHECKPOINT explícito')
if RESUME_CHECKPOINT and not Path(RESUME_CHECKPOINT).is_file(): raise FileNotFoundError(RESUME_CHECKPOINT)
PREFLIGHT = {'commit': GIT_COMMIT, 'runtime': RUNTIME, 'hdf5': [MUTANTS_RECORD, WT_RECORD], 'hdf5_validation': HDF5_VALIDATION, 'output': OUTPUT_PREFLIGHT, 'config': str(RESOLVED_CONFIG)}
print(json.dumps(PREFLIGHT, indent=2, default=str))


## 7. Smoke inicial + resume productivo y validación de artefactos

In [ ]:
from scripts.colab_preflight import parse_cli_contract, run_command
SMOKE_KEYS = ('manifest_status','resume_manifest_status','epochs_completed','resume_epochs_completed','global_step','resume_global_step','best_checkpoint','last_checkpoint','resume_last_checkpoint','run_dir','resume_run_dir','manifest_path','resume_manifest_path')
SMOKE_RESULT = None
if EXECUTION_MODE == 'smoke':
    command = build_train_command(repo, RESOLVED_CONFIG, device=REQUESTED_DEVICE, smoke_test=True)
    SESSION_RECORD['commands'].append(command); write_session_record(SESSION_RECORD_PATH, allowed_root=OUTPUT_ROOT, payload=SESSION_RECORD)
    print('command=', command)
    SMOKE_RESULT = run_command(command, cwd=repo)
    print(SMOKE_RESULT.stdout)
    if SMOKE_RESULT.stderr: print(SMOKE_RESULT.stderr, file=sys.stderr)
    if SMOKE_RESULT.returncode != 0: raise subprocess.CalledProcessError(SMOKE_RESULT.returncode, command, SMOKE_RESULT.stdout, SMOKE_RESULT.stderr)
    smoke_fields = parse_cli_contract(SMOKE_RESULT.stdout, required_keys=SMOKE_KEYS, allowed_keys=SMOKE_KEYS)
    assert smoke_fields['manifest_status'] == smoke_fields['resume_manifest_status'] == 'completed'
    assert int(smoke_fields['epochs_completed']) == 1 and int(smoke_fields['resume_epochs_completed']) == 2
    assert int(smoke_fields['global_step']) == 2 and int(smoke_fields['resume_global_step']) == 4
    from gnn_siamese.training import load_checkpoint
    for key in ('best_checkpoint','last_checkpoint','resume_last_checkpoint'):
        payload = load_checkpoint(smoke_fields[key]); assert payload['format_version'] == 1; assert payload['compatibility']['compatibility_metadata']['version'] == 2
    print({'initial_run': smoke_fields['run_dir'], 'resumed_run': smoke_fields['resume_run_dir'], 'manifest': smoke_fields['manifest_path']})


## 8. Piloto (protegido; desactivado por defecto)

In [ ]:
PILOT_RESULT = None
if EXECUTION_MODE == 'pilot' and RUN_PILOT:
    command = build_train_command(repo, RESOLVED_CONFIG, device=REQUESTED_DEVICE)
    SESSION_RECORD['commands'].append(command); write_session_record(SESSION_RECORD_PATH, allowed_root=OUTPUT_ROOT, payload=SESSION_RECORD)
    print('command=', command); PILOT_RESULT = run_command(command, cwd=repo); print(PILOT_RESULT.stdout)
    if PILOT_RESULT.stderr: print(PILOT_RESULT.stderr, file=sys.stderr)
    if PILOT_RESULT.returncode != 0: raise subprocess.CalledProcessError(PILOT_RESULT.returncode, command)
    pilot_keys = ('run_dir','last_checkpoint','epochs_completed')
    pilot_fields = parse_cli_contract(PILOT_RESULT.stdout, required_keys=pilot_keys, allowed_keys=pilot_keys)
    if not Path(pilot_fields['last_checkpoint']).is_file(): raise RuntimeError('No apareció last.pt')


## 9. Resume del piloto (checkpoint explícito; run nuevo)

In [ ]:
RESUME_RESULT = None
if EXECUTION_MODE == 'resume' and RUN_RESUME:
    checkpoint = Path(RESUME_CHECKPOINT).resolve()
    if not checkpoint.is_file(): raise FileNotFoundError(f'Checkpoint explícito inexistente: {checkpoint}')
    source_manifest = checkpoint.parent.parent / 'run_manifest.json'
    if not source_manifest.is_file(): raise FileNotFoundError(f'Manifiesto de origen inexistente: {source_manifest}')
    print(json.loads(source_manifest.read_text(encoding='utf-8')))
    command = build_train_command(repo, RESOLVED_CONFIG, device=REQUESTED_DEVICE, resume_from=checkpoint)
    SESSION_RECORD['commands'].append(command); write_session_record(SESSION_RECORD_PATH, allowed_root=OUTPUT_ROOT, payload=SESSION_RECORD)
    print('command=', command); RESUME_RESULT = run_command(command, cwd=repo); print(RESUME_RESULT.stdout)
    if RESUME_RESULT.stderr: print(RESUME_RESULT.stderr, file=sys.stderr)
    if RESUME_RESULT.returncode != 0: raise subprocess.CalledProcessError(RESUME_RESULT.returncode, command)
    resume_keys = ('run_dir','last_checkpoint','epochs_completed')
    resume_fields = parse_cli_contract(RESUME_RESULT.stdout, required_keys=resume_keys, allowed_keys=resume_keys)
    if Path(resume_fields['run_dir']).resolve() == source_manifest.parent.resolve(): raise RuntimeError('Resume sobrescribió el run de origen')
    print({'source_epoch': json.loads(source_manifest.read_text())['training']['epochs_completed'], 'final_epoch': resume_fields['epochs_completed']})


## 10. Sincronización explícita y resumen final

In [ ]:
if OUTPUT_MODE == 'local_sync' and SYNC_LOCAL_OUTPUTS:
    destination = DRIVE_SYNC_ROOT; destination.mkdir(parents=True, exist_ok=True)
    from scripts.colab_preflight import sync_local_outputs
    sync = sync_local_outputs(OUTPUT_ROOT, destination)
fields = locals().get('resume_fields', locals().get('pilot_fields', locals().get('smoke_fields', {})))
FINAL_SUMMARY = {'commit': GIT_COMMIT, 'device': RUNTIME['selected_device'], 'hdf5': [MUTANTS_RECORD, WT_RECORD], 'config': str(RESOLVED_CONFIG),
 'run_id': fields.get('run_dir','').split('run_')[-1] if fields else None, 'status': fields.get('resume_manifest_status', fields.get('manifest_status')),
 'epoch_completed': fields.get('resume_epochs_completed', fields.get('epochs_completed')), 'global_step': fields.get('resume_global_step', fields.get('global_step')),
 'best_checkpoint': fields.get('best_checkpoint'), 'last_checkpoint': fields.get('resume_last_checkpoint', fields.get('last_checkpoint')),
 'output_root': str(OUTPUT_ROOT), 'next_action': 'activar RUN_PILOT deliberadamente' if not RUN_PILOT else 'seleccionar last.pt explícito para resume'}
print(json.dumps(FINAL_SUMMARY, indent=2, default=str))
